# 22.8 — Automated planning (STRIPS, PDDL)

STRIPS planning turns actions into preconditions, add effects, and delete effects, then searches over fact sets.

Graph search supplies the engine and STRIPS supplies symbolic states. The successor function applies preconditions, add lists, and delete lists to fact sets.

Save a copy to Drive to edit.

In [ ]:
from collections import deque
from dataclasses import dataclass
from typing import FrozenSet, List, Set, Tuple

import matplotlib.pyplot as plt


@dataclass(frozen=True)
class Action:
    name: str
    pre: FrozenSet[str]
    add: FrozenSet[str]
    delete: FrozenSet[str]


def apply_action(state, action):
    if not action.pre.issubset(state):
        return None
    return frozenset((set(state) - set(action.delete)) | set(action.add))


def regress(goal, action):
    return frozenset((set(goal) - set(action.add)) | set(action.pre))


def bfs_plan(initial, goal, actions):
    queue = deque([(initial, [])])
    seen = {initial}
    expanded = 0
    while queue:
        state, plan = queue.popleft()
        expanded += 1
        if goal.issubset(state):
            return plan, expanded, seen
        for action in actions:
            nxt = apply_action(state, action)
            if nxt is None:
                continue
            if nxt in seen:
                continue
            seen.add(nxt)
            queue.append((nxt, plan + [action.name]))
    return None, expanded, seen


def move(a, b):
    return Action(f"Move{a}{b}", frozenset({f"At{a}", f"Road{a}{b}"}), frozenset({f"At{b}"}), frozenset({f"At{a}"}))


def make_road_domain(name, locations, edges, packages=0, interacting=False):
    actions = []
    roads = set()
    for a, b in edges:
        actions.append(move(a, b))
        roads.add(f"Road{a}{b}")
    initial = frozenset({"AtA"} | roads)
    goal = frozenset({f"At{locations[-1]}"})
    for package in range(packages):
        pickup = Action(f"PickP{package}", frozenset({"AtB", f"Pkg{package}AtB"}), frozenset({f"HaveP{package}"}), frozenset({f"Pkg{package}AtB"}))
        drop = Action(f"DropP{package}", frozenset({f"At{locations[-1]}", f"HaveP{package}"}), frozenset({f"DeliveredP{package}"}), frozenset({f"HaveP{package}"}))
        actions.extend([pickup, drop])
        initial = frozenset(set(initial) | {f"Pkg{package}AtB"})
        goal = frozenset(set(goal) | {f"DeliveredP{package}"})
    if interacting:
        actions.append(Action("ShortcutWithStaleAtA", frozenset({"AtA"}), frozenset({f"At{locations[-1]}"}), frozenset()))
    return {"name": name, "initial": initial, "goal": goal, "actions": actions}


def strips_ladder():
    return [
        make_road_domain("D1 three-location", ["A", "B", "C"], [("A", "B"), ("B", "C")]),
        make_road_domain("D2 wider roads", ["A", "B", "C", "D"], [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D")]),
        make_road_domain("D3 package delivery", ["A", "B", "C", "D"], [("A", "B"), ("B", "C"), ("C", "D")], packages=1),
        make_road_domain("D4 interacting deletes", ["A", "B", "C", "D", "E"], [("A", "B"), ("B", "C"), ("C", "E"), ("B", "D"), ("D", "E")], packages=2),
        make_road_domain("D5 deceptive stale facts", ["A", "B", "C", "D", "E", "F"], [("A", "B"), ("B", "C"), ("C", "D"), ("D", "E"), ("E", "F"), ("B", "E")], packages=2, interacting=True),
    ]

## The concept, built once (D1)

An action applies when $Pre(a)\subseteq S$, and the successor is $S'=(S\setminus Del(a))\cup Add(a)$.

In [ ]:
state = frozenset({"AtA", "RoadAB"})
action = move("A", "B")
successor = apply_action(state, action)
print(successor)
assert successor == frozenset({"RoadAB", "AtB"})
assert len(successor) == 2

Forward planning to $AtC$ uses BFS over fact sets. Regression through $MoveBC$ transforms goal $\{AtC\}$ into $\{AtB,RoadBC\}$.

In [ ]:
initial = frozenset({"AtA", "RoadAB", "RoadBC"})
goal = frozenset({"AtC"})
actions = [move("A", "B"), move("B", "C")]
plan, expanded, seen = bfs_plan(initial, goal, actions)
regressed = regress(goal, move("B", "C"))
print("plan", plan, "expanded", expanded)
print("regressed", regressed)
assert len(plan) == 2
assert regressed == frozenset({"AtB", "RoadBC"})
assert len(regressed) == 2

## The dataset ladder

D1–D5 grow from a three-location domain to package delivery, interacting deletes, and a deceptive stale-fact shortcut.

In [ ]:
ladder = strips_ladder()
for problem in ladder:
    print(problem["name"], "facts", len(problem["initial"] | problem["goal"]), "actions", len(problem["actions"]), "goal", sorted(problem["goal"]))
    print("sample actions", [action.name for action in problem["actions"][:4]])

## Run the SAME method across D1–D5

Metric: legal plan length and expanded fact states.

In [ ]:
results = []
for problem in ladder:
    plan, expanded, seen = bfs_plan(problem["initial"], problem["goal"], problem["actions"])
    length = len(plan) if plan else None
    results.append({"name": problem["name"], "length": length, "expanded": expanded, "states": len(seen), "plan": plan})
print("rung | plan length | expanded | seen")
for row in results:
    print(row["name"], row["length"], row["expanded"], row["states"], row["plan"])

## Results visualization

Panels show plan length and expanded fact states. The summary curve tracks solution size and search effort.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for axis, row in zip(axes, results):
    axis.bar(["length", "expanded"], [row["length"] or 0, row["expanded"]], color=["darkorange", "steelblue"])
    axis.set_title(row["name"].split()[0])
plt.show()
xs = list(range(1, len(results) + 1))
fig, axis = plt.subplots(figsize=(6, 3))
axis.plot(xs, [row["length"] or 0 for row in results], marker="o", label="plan length")
axis.plot(xs, [row["expanded"] for row in results], marker="s", label="expanded")
axis.legend()
axis.set_xlabel("D rung")
plt.show()

## Pitfall on D5: forgetting delete effects

Without delete effects, stale facts can make the planner believe mutually exclusive locations are true. The fix is the exact $(S\setminus Del)\cup Add$ successor.

In [ ]:
bad_state = frozenset({"AtA", "RoadAB"})
bad_action = Action("BadMoveAB", frozenset({"AtA", "RoadAB"}), frozenset({"AtB"}), frozenset())
stale = apply_action(bad_state, bad_action)
fixed = apply_action(bad_state, move("A", "B"))
print("without deletes", sorted(stale))
print("with deletes", sorted(fixed))
assert "AtA" in stale and "AtB" in stale
assert "AtA" not in fixed and "AtB" in fixed

## Evaluate it + Practice

- Metric: plan length and expanded nodes.
- No-skill baseline: ignore deletes or greedily chase one goal fact.
- Cheap sanity check: MoveAB keeps fact count 2 and D1 plan length is 2.
- Ablation: remove delete effects and stale facts appear.
- Failure signals: plans contain impossible simultaneous locations or unexplained missing preconditions.

### Practice
Add a package to D3 and compare expanded states.

### Practice
Write a relaxed goal-count heuristic and find where it misleads D5.

### Practice
Add reverse roads and compare shortest plan length.